#Instantiate Spark Session 

In [1]:
import findspark
findspark.init()

In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("pyspark_assesment").getOrCreate()

#Read Data

In [13]:
def read_csv(path,header=True,inferSchema=True,sep=','):
    df = spark.read.csv(path, header=header, inferSchema=True,sep=sep)
    return df

#Data Exploration(EDA) - Data Munging<br>
Passive Munging - Visually analysing the data<br>
Active Munging - Schema Evolution, validation, standardization and de-duplication<br>

In [ ]:
def writeFile(df,file_path,file_format,mode='Overwrite'):
    df.write.mode(mode).format(file_format).save(file_path)

##Passive Munging

Passive Munging:<br>
    1. File has header<br>
    2. No footer<br>
    3. File is comma separator<br>
    4. Blank Values in job category and full/time indicator<br>
    5. Data Standardization needed for Business Title because of small/capital letters<br>
    6. DateTimestamp columns is considered as String<br>
    7. Data duplication<br>
    

In [1]:
def passive_Munging(df):
    print(df.columns) # returns the list of columns
    print(df.dtypes) # returns column names with datatype
    print(df.schema)  # returns the schema in StructType format
    print("Total Number of Records -",df.count())
    print("Total Number of Distinct records -", df.distinct().count())
    print("Total Number of Duplicate records",df.count()-df.distinct().count())

##Active Munging - Schema Evolution, validation, standardization and de-duplication

In [2]:
###Write the DataFrame in orc/parquet format for optimization and to perform schema evolution
###Validation- Applying Strict Schema to the DataFrame based on the Business requirement and capturing corrupt rows
###Write corrupt record to the file

    
def Active_Munging(file_path,schema,corrupt_column, Primary_key ,mode='Overwrite'):
    df=spark.read.schema(schema).option("mergeSchema","True").option("columnNameOfCorruptRecord",corrupt_column).orc(file_path)
    df.cache()
    corrupt_record=df.filter(f"{corrupt_column} is not null") # we can save it in dataset and share it with Business for data correction
    if corrupt_record.count() > 0:
        writeFile(corrupt_record,"/dataset/reject/","csv")
    return df

#Data Enrichment - converting string type to timestamp

In [11]:
###Standardization 
#1.delete the rows if it contains NULL across all the columns and 
#2.delete the rows if any of the Job ID contains NULL
#3.Removes duplicates
from pyspark.sql.functions import *

def data_enrichment(df,col_list,Primary_key):
    for column in col_list:
        df1=df.withColumn(column,to_timestamp(col(column)))
        df_enrich=df1.na.drop(how='all').na.drop(subset=[f"{Primary_key}"],how='any').na.fill("N/A").distinct()
    return df_enrich

   
    